# Limpeza e Consolidação de Dados do FUNDEB (2013-2024)
Este notebook realiza a leitura, padronização e filtragem das transferências obrigatórias e voluntárias do FUNDEB para a Baixada Fluminense.


In [ ]:
import polars as pl
import os
import io
import glob
import re

BAIXADA_MUNICIPIOS = [
    'BELFORD ROXO', 'DUQUE DE CAXIAS', 'GUAPIMIRIM', 'ITAGUAI', 'ITAGUAÍ', 
    'JAPERI', 'MAGE', 'MAGÉ', 'MESQUITA', 'NILOPOLIS', 'NILÓPOLIS', 
    'NOVA IGUACU', 'NOVA IGUAÇU', 'PARACAMBI', 'QUEIMADOS', 
    'SAO JOAO DE MERITI', 'SÃO JOÃO DE MERITI', 'SEROPEDICA', 'SEROPÉDICA'
]

# Normalização de nomes com acento e prefixos de prefeituras
def normalizar_mun(m):
    if not m: return None
    m = str(m).upper().strip()
    m = re.sub(r'^PREFEITURA\s+MUNICIPAL\s+DE\s+', '', m)
    m = re.sub(r'^PREFEITURA\s+DE\s+', '', m)
    m = m.replace('ITAGUAÍ', 'ITAGUAI').replace('MAGÉ', 'MAGE').replace('NILÓPOLIS', 'NILOPOLIS').replace('NOVA IGUAÇU', 'NOVA IGUACU').replace('SÃO JOÃO DE MERITI', 'SAO JOAO DE MERITI').replace('SEROPÉDICA', 'SEROPEDICA')
    # Variantes de São João de Meriti
    m = m.replace('SÃO JOAO DE MERITI', 'SAO JOAO DE MERITI')
    m = m.replace('S.JOÃO DE MERITI', 'SAO JOAO DE MERITI')
    m = m.replace('S.JOAO DE MERITI', 'SAO JOAO DE MERITI')
    m = m.replace('S. JOÃO DE MERITI', 'SAO JOAO DE MERITI')
    m = m.replace('S. JOAO DE MERITI', 'SAO JOAO DE MERITI')
    return m

def get_year_from_filename(f):
    m = re.search(r'20[12][0-9]', f)
    if m: return int(m.group())
    return None

def clean_currency_expr(col_name):
    # Regex para remover R$, espaços e pontuação para deixar formato float
    return (
        pl.col(col_name).cast(pl.Utf8)
        .str.replace(r'R\$\s*', '')
        .str.replace_all(r'\.', '')
        .str.replace(r',', '.')
        .str.strip_chars()
        .str.replace(r'^\-$', '0')
        .cast(pl.Float64, strict=False)
        .fill_null(0.0)
    )


In [ ]:
# 1. Transferências Obrigatórias
obrig_path = '../raw_data/Municipios_FUNDEB/Transferencias_Obrigatorias/'
files = sorted(glob.glob(os.path.join(obrig_path, '*.csv')))

dfs_obrig = []

for file in files:
    ano = get_year_from_filename(file)
    if not ano or ano < 2011: continue
    print(f"Processando {file} (Ano {ano})...")
    
    with open(file, 'r', encoding='latin1') as f:
        content = f.read()
        
    lines = content.split('\n')
    header_idx = None
    for i, line in enumerate(lines):
        parts = [p.upper().strip().replace('"', '') for p in line.split(';')]
        has_mun = any('MUNIC' in p or 'PREFEITURA' in p for p in parts)
        has_key = any('TRIBUTO' in p or 'TOTAL' in p or 'JANEIRO' in p or 'VALOR' in p or 'DATA' in p for p in parts)
        non_empty_parts = [p for p in parts if p != '']
        if has_mun and has_key and len(non_empty_parts) > 2:
            header_idx = i
            break
            
    if header_idx is None:
        print(f"Aviso: Cabeçalho não encontrado para {file}. Pulando...")
        continue
        
    clean_csv = '\n'.join(lines[header_idx:])
    df = pl.read_csv(io.BytesIO(clean_csv.encode('utf-8')), separator=';', infer_schema_length=0, ignore_errors=True)
    
    new_cols = []
    seen = set()
    for c in df.columns:
        name = str(c).upper().strip()
        if name in seen:
            j = 1
            while f"{name}_{j}" in seen:
                j += 1
            name = f"{name}_{j}"
        seen.add(name)
        new_cols.append(name)
    df.columns = new_cols
    
    mun_cols = [c for c in df.columns if 'MUNIC' in c or 'PREFEITURA' in c]
    if not mun_cols:
        print(f"Aviso: Coluna de município não encontrada em {file}. Pulando...")
        continue
    mun_col = mun_cols[0]
    
    # Preenche a coluna do município usando forward fill com nomes normalizados
    df = df.with_columns(
        pl.when(pl.col(mun_col).cast(pl.Utf8).str.strip_chars() == '')
        .then(None)
        .otherwise(
            pl.col(mun_col).cast(pl.Utf8)
            .str.to_uppercase()
            .str.strip_chars()
            .str.replace(r'^PREFEITURA\s+MUNICIPAL\s+DE\s+', '', literal=False)
            .str.replace(r'^PREFEITURA\s+DE\s+', '', literal=False)
            .str.replace('ITAGUAÍ', 'ITAGUAI')
            .str.replace('MAGÉ', 'MAGE')
            .str.replace('NILÓPOLIS', 'NILOPOLIS')
            .str.replace('NOVA IGUAÇU', 'NOVA IGUACU')
            .str.replace('SÃO JOÃO DE MERITI', 'SAO JOAO DE MERITI')
            .str.replace('SEROPÉDICA', 'SEROPEDICA')
            # Variantes de São João de Meriti
            .str.replace('SÃO JOAO DE MERITI', 'SAO JOAO DE MERITI')
            .str.replace('S.JOÃO DE MERITI', 'SAO JOAO DE MERITI')
            .str.replace('S.JOAO DE MERITI', 'SAO JOAO DE MERITI')
            .str.replace('S. JOÃO DE MERITI', 'SAO JOAO DE MERITI')
            .str.replace('S. JOAO DE MERITI', 'SAO JOAO DE MERITI')
        )
        .fill_null(strategy="forward")
        .alias('mun_clean')
    )
        
    df = df.filter(pl.col('mun_clean').is_not_null())
    df = df.filter(pl.col('mun_clean').cast(pl.Utf8).str.strip_chars() != '')
    
    # Filtro da Baixada Fluminense
    baixada_norms = [normalizar_mun(m) for m in BAIXADA_MUNICIPIOS]
    df = df.filter(pl.col('mun_clean').is_in(baixada_norms))
    if df.height == 0: continue
    
    trib_cols = [c for c in df.columns if 'TRIBUTO' in c]
    if trib_cols:
        trib_col = trib_cols[0]
    else:
        df = df.with_columns(pl.lit("UNKNOWN").alias("TRIBUTO"))
        trib_col = "TRIBUTO"
    
    total_col = [c for c in df.columns if 'TOTAL' in c]
    if total_col:
        df = df.with_columns(clean_currency_expr(total_col[0]).alias('valor_total'))
    else:
        ignore_cols = [mun_col, trib_col, 'REGIAO', 'CNPJ', 'REGIÃO', 'MUN_CLEAN']
        val_cols = [c for c in df.columns if c not in ignore_cols and 'UNNAMED' not in c]
        cleaned_cols = [clean_currency_expr(c) for c in val_cols]
        df = df.with_columns(pl.sum_horizontal(cleaned_cols).alias('valor_total'))
        
    df = df.with_columns(
        pl.col(trib_col).cast(pl.Utf8).str.to_uppercase().str.contains('FUNDEB').fill_null(False).alias('is_fundeb')
    )
    df = df.with_columns(
        pl.when(pl.col('is_fundeb')).then(pl.col('valor_total')).otherwise(0.0).alias('valor_fundeb')
    )
    
    agrup = df.group_by('mun_clean').agg([
        pl.col('valor_total').sum().alias('total_obrig'),
        pl.col('valor_fundeb').sum().alias('total_fundeb')
    ])
    
    agrup = agrup.with_columns(pl.lit(int(ano)).alias('ano'))
    dfs_obrig.append(agrup)

if dfs_obrig:
    try:
        df_obrig_final = pl.concat(dfs_obrig, how="diagonal_relaxed")
    except:
        df_obrig_final = pl.concat(dfs_obrig, how="diagonal")
else:
    df_obrig_final = pl.DataFrame()
    
print("Obrigatórias processadas.")


In [ ]:
# 2. Transferências Voluntárias
vol_path = '../raw_data/Municipios_FUNDEB/Transferencias_Voluntarias/Copia-de-Transferencias-Voluntarias-aos-Municipios-2015-a-2025.csv'

with open(vol_path, 'r', encoding='latin1') as f:
    lines = f.readlines()
    
header_idx = 0
for i, line in enumerate(lines[:10]):
    if 'FAVORECIDO' in line.upper():
        header_idx = i
        break

df_vol = pl.read_csv(vol_path, separator=';', skip_rows=header_idx, encoding='iso-8859-1', infer_schema_length=0, ignore_errors=True)
cols = df_vol.columns
cols[0] = 'ANO'
new_cols = []
seen = set()
for c in cols:
    name = str(c).upper().strip()
    if name in seen:
        j = 1
        while f"{name}_{j}" in seen:
            j += 1
        name = f"{name}_{j}"
    seen.add(name)
    new_cols.append(name)
df_vol.columns = new_cols

def extract_municipio(text):
    if text is None: return None
    normalized_text = normalizar_mun(str(text))
    if normalized_text is None: return None
    for m in BAIXADA_MUNICIPIOS:
        norm_m = normalizar_mun(m)
        if norm_m in normalized_text:
            return norm_m
    return None

df_vol = df_vol.with_columns(
    pl.col('FAVORECIDO').map_elements(extract_municipio, return_dtype=pl.Utf8).alias('mun_clean')
)
df_vol = df_vol.filter(pl.col('mun_clean').is_not_null() & pl.col('ANO').is_not_null())

df_vol = df_vol.with_columns(pl.col('ANO').cast(pl.Int32, strict=False))
df_vol = df_vol.filter(pl.col('ANO').is_not_null())

if 'TOTAL' in df_vol.columns:
    df_vol = df_vol.with_columns(clean_currency_expr('TOTAL').alias('valor_voluntario'))
else:
    df_vol = df_vol.with_columns(pl.lit(0.0).alias('valor_voluntario'))

agrup_vol = df_vol.group_by(['mun_clean', 'ANO']).agg([
    pl.col('valor_voluntario').sum().alias('total_voluntario')
])

agrup_vol = agrup_vol.rename({'ANO': 'ano'})
print("Voluntárias processadas.")


In [ ]:
# 3. Merge e Exportação com Deflação IPCA (valores reais de Dez/2024)
if not df_obrig_final.is_empty():
    try:
        df_final = df_obrig_final.join(agrup_vol, on=['mun_clean', 'ano'], how='full', coalesce=True)
    except TypeError:
        # Polars < 1.0 não tem coalesce=True
        df_final = df_obrig_final.join(agrup_vol, on=['mun_clean', 'ano'], how='outer')
else:
    df_final = agrup_vol

# Se o merge gerou nulls
df_final = df_final.fill_null(0.0)

# Preencher possíveis nulls nas chaves de município e ano causados por outer join 
if 'mun_clean_right' in df_final.columns:
    df_final = df_final.with_columns(pl.col('mun_clean').fill_null(pl.col('mun_clean_right')))
if 'ano_right' in df_final.columns:
    df_final = df_final.with_columns(pl.col('ano').fill_null(pl.col('ano_right')))

df_final = df_final.with_columns(
    (pl.col('total_obrig') + pl.col('total_voluntario')).alias('total_geral')
)

df_final = df_final.rename({'mun_clean': 'municipio'})

# --- DEFLAÇÃO IPCA ---
import urllib.request
import json as py_json
import pandas as pd

url = 'https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados?formato=json&dataInicial=01/01/2011&dataFinal=31/12/2024'
req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
try:
    with urllib.request.urlopen(req) as response:
        ipca_data = py_json.loads(response.read().decode('utf-8'))
    df_ipca = pd.DataFrame(ipca_data)
    df_ipca['valor'] = pd.to_numeric(df_ipca['valor'])
    df_ipca['fator'] = 1 + (df_ipca['valor'] / 100)
    df_ipca['ind'] = df_ipca['fator'].cumprod()
    df_ipca['data'] = pd.to_datetime(df_ipca['data'], format='%d/%m/%Y')
    df_ipca['ano'] = df_ipca['data'].dt.year
    df_ipca['mes'] = df_ipca['data'].dt.month
    
    ref_val = df_ipca[(df_ipca['ano'] == 2024) & (df_ipca['mes'] == 12)]['ind'].values[0]
    df_ano = df_ipca.groupby('ano')['ind'].mean().reset_index()
    df_ano['fator_deflator'] = ref_val / df_ano['ind']
    fatores_dict = dict(zip(df_ano['ano'], df_ano['fator_deflator']))
    print("Fatores IPCA obtidos com sucesso do Banco Central.")
except Exception as e:
    print("Erro ao acessar API do BCB. Usando fatores estáticos de backup.", e)
    fatores_dict = {
        2011: 2.137938, 2012: 2.028334, 2013: 1.909844, 2014: 1.796161,
        2015: 1.647399, 2016: 1.514995, 2017: 1.464523, 2018: 1.412750,
        2019: 1.361910, 2020: 1.319532, 2021: 1.218386, 2022: 1.114922,
        2023: 1.065957, 2024: 1.021351
    }

df_fatores = pl.DataFrame([
    {'ano': float(k), 'fator_deflator': float(v)} for k, v in fatores_dict.items()
])

df_final = df_final.join(df_fatores, on='ano', how='left').fill_null(1.0)

# Aplicar fator de correção sobre as colunas nominais do FUNDEB
df_final = df_final.with_columns([
    (pl.col('total_obrig') * pl.col('fator_deflator')).alias('total_obrig'),
    (pl.col('total_fundeb') * pl.col('fator_deflator')).alias('total_fundeb'),
    (pl.col('total_voluntario') * pl.col('fator_deflator')).alias('total_voluntario'),
    (pl.col('total_geral') * pl.col('fator_deflator')).alias('total_geral')
]).drop('fator_deflator')

pl.Config.set_fmt_float('full')
display(df_final)

os.makedirs('../curated/parquet/fundeb', exist_ok=True)
output_path = '../curated/parquet/fundeb/dataset_fundeb_municipio_ano.parquet'

df_final.write_parquet(output_path)
print(f"Dados consolidados REAIS (deflacionados IPCA a dez/24) salvos em {output_path}")
